# Эксперименты на Amazon

Подключите версию данных с каталогом, сплитами и манифестом. Для Kaggle добавьте исходный снимок проекта как вход для ноутбука и задайте пути ниже. Локально можно указать корень репозитория и локальный split. Первый запуск по умолчанию оценивается на validation.

Этот ноутбук запускает одну партию на validation или test; для выбора гиперпараметров и повторов по трём seed используйте scripts/run_study.py по инструкции README.

В таблице результатов HitRate@10 — доля запросов с хотя бы одним релевантным товаром в топ-10.

По умолчанию используются модели и бейзлайны из configs/amazon_train_only.json.

## Результаты готовой серии

Ниже сохранены результаты Amazon test при 100%, 50% и 25% обучающих покупок. Метрики усреднены по трём seed. Таблицы читаются из [test_summary.csv](../docs/results/amazon-train-only/test_summary.csv). Для их просмотра обучение и исходные данные не нужны.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

relative_path = Path("docs/results/amazon-train-only/test_summary.csv")
project_roots = (
    Path(os.environ.get("RECSYS_PROJECT_ROOT", "/kaggle/input/complementary-recsys-code")),
    Path.cwd(),
    Path.cwd().parent,
)
summary_file = next(
    (root / relative_path for root in project_roots if (root / relative_path).is_file()),
    None,
)
if summary_file is None:
    raise FileNotFoundError(f"Укажите RECSYS_PROJECT_ROOT с файлом {relative_path}")

summary = pd.read_csv(summary_file)
metrics = {"hit_rate_mean": "HitRate@10", "ndcg_mean": "NDCG@10", "recall_mean": "Recall@10"}
for buy_fraction, selected in summary.groupby("buy_fraction", sort=False):
    display(Markdown(f"**{buy_fraction:.0%} обучающих покупок**"))
    display(
        selected.sort_values(list(metrics), ascending=False)
        .set_index("model")[list(metrics)]
        .rename(columns=metrics)
        .rename_axis("Модель")
        .round(4)
    )

**100% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.7474,0.5249,0.7248
CP buy-only,0.7061,0.4615,0.6800
CP buy+view,0.6980,0.4505,0.6722
Tucker buy-only,0.6938,0.4444,0.6669
MIG-TF,0.6916,0.4458,0.6650
ComplEx buy+view,0.6916,0.4454,0.6651
DistMult buy-only,0.6910,0.4504,0.6653
ComplEx buy-only,0.6904,0.4495,0.6641
Tucker buy+view,0.6867,0.4393,0.6604


**50% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.6600,0.4498,0.6350
MIG-TF,0.6280,0.3917,0.6001
DistMult buy+view,0.6233,0.3926,0.5954
Tucker buy+view,0.6200,0.3853,0.5926
DistMult buy-only,0.6199,0.3936,0.5916
Tucker buy-only,0.6172,0.3825,0.5898
TwoTower Mixed,0.6144,0.3789,0.5869
ComplEx buy+view,0.6113,0.3834,0.5850
CP buy+view,0.6090,0.3823,0.5824


**25% обучающих покупок**

,HitRate@10,NDCG@10,Recall@10
Модель,,,
HOSVD,0.6185,0.4044,0.5941
TwoTower Mixed,0.6111,0.3716,0.5858
MIG-TF,0.5427,0.3242,0.5173
Tucker buy+view,0.5316,0.3148,0.5061
Tucker buy-only,0.5246,0.3093,0.4992
DistMult buy+view,0.4702,0.2894,0.4455
DistMult buy-only,0.4609,0.2837,0.4362
ComplEx buy-only,0.4107,0.2435,0.3878
CP buy-only,0.4056,0.2413,0.3827


## Запуск нового эксперимента

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path(os.environ.get('RECSYS_PROJECT_ROOT', '/kaggle/input/complementary-recsys-code'))
SPLIT_DIR = Path(os.environ.get('RECSYS_SPLIT_DIR', '/kaggle/input/amazon-complementary-split'))
OUTPUT_ROOT = Path(os.environ.get('RECSYS_OUTPUT_ROOT', '/kaggle/working/runs'))
CODE_REVISION = os.environ.get('RECSYS_CODE_REVISION')
CONFIG = os.environ.get('RECSYS_CONFIG', 'configs/amazon_train_only.json')
PARTITION = os.environ.get('RECSYS_PARTITION', 'validation')

assert (PROJECT_ROOT / 'src/complementary_recsys').is_dir(), PROJECT_ROOT
assert (SPLIT_DIR / 'manifest.json').is_file(), SPLIT_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import torch
from complementary_recsys.data.bundle import SplitBundle
from complementary_recsys.experiment import ExperimentRunner

config = json.loads((PROJECT_ROOT / CONFIG).read_text())
config['training']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
if CODE_REVISION:
    config['code_revision'] = CODE_REVISION
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = OUTPUT_ROOT / run_id
bundle = SplitBundle.load(SPLIT_DIR)
results = ExperimentRunner(bundle, config, PARTITION).run(output)
display(results)

In [ ]:
import pandas as pd

recommendations = pd.read_csv(output / 'recommendations.csv.gz', dtype={'source_id': str})
display(recommendations.head(10))
print(output)